## Understand the difference between mask-based tumour-positive case count against metadata's `tumor?` column.

### Goal: identify exactly which cases disagree (mask-labeling, metadata, or a case-ID mismatch)

In [ ]:
import os
import numpy as np
import pandas as pd
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from config import LABEL_DIR
from monai.transforms import LoadImage

## Rebuild mask-based positive set with case IDs

In [1]:
loader = LoadImage(image_only=True)

cases = sorted(os.listdir(LABEL_DIR))
lesion_counts = {}

for case in cases:
    lesion_path = f"{LABEL_DIR}/{case}/segmentations/pancreatic_lesion.nii.gz"
    if os.path.exists(lesion_path):
        data = loader(lesion_path).numpy()
        lesion_counts[case] = np.count_nonzero(data)

mask_positive_cases = set(k for k, v in lesion_counts.items() if v > 0)
mask_negative_cases = set(k for k, v in lesion_counts.items() if v == 0)
mask_missing_lesion_file = set(cases) - set(lesion_counts.keys())

print(f"Mask-based total cases with lesion file: {len(lesion_counts)}")
print(f"Mask-based tumour-positive: {len(mask_positive_cases)}")
print(f"Mask-based tumour-negative: {len(mask_negative_cases)}")
print(f"Cases missing lesion file entirely: {len(mask_missing_lesion_file)}")

Mask-based total cases with lesion file: 9901
Mask-based tumor-positive: 1033
Mask-based tumor-negative: 8868
Cases missing lesion file entirely: 1


## Get metadata-based positive set

In [2]:
data_root = "/Volumes/BackupDrive/pants/data"
metadata = pd.read_excel(f"{data_root}/metadata.xlsx")

print("\nMetadata columns:", list(metadata.columns))

CASE_ID_COLUMN = "PanTS ID"

metadata_positive_cases = set(
    metadata.loc[metadata["tumor?"] == 1, CASE_ID_COLUMN].astype(str)
)
metadata_negative_cases = set(
    metadata.loc[metadata["tumor?"] == 0, CASE_ID_COLUMN].astype(str)
)

print(f"\nMetadata-based tumour-positive: {len(metadata_positive_cases)}")
print(f"Metadata-based tumour-negative: {len(metadata_negative_cases)}")



Metadata columns: ['PanTS ID', 'shape', 'spacing', 'ct phase', 'sex', 'age', 'manufacturer', 'manufacturer model', 'study type', 'site', 'site detail', 'site nationality', 'study year', 'tumor?', 'structured report']

Metadata-based tumor-positive: 1077
Metadata-based tumor-negative: 8824


## Find the actual discrepancy

In [3]:
metadata_says_positive_mask_says_not = metadata_positive_cases - mask_positive_cases

mask_says_positive_metadata_says_not = mask_positive_cases - metadata_positive_cases

cases_in_metadata_not_in_masks = metadata_positive_cases.union(metadata_negative_cases) - set(cases)
cases_in_masks_not_in_metadata = set(cases) - metadata_positive_cases.union(metadata_negative_cases)

print(f"\n--- Discrepancy breakdown ---")
print(f"Metadata says positive, mask says negative/missing: {len(metadata_says_positive_mask_says_not)}")
print(f"Mask says positive, metadata says negative: {len(mask_says_positive_metadata_says_not)}")
print(f"Case IDs in metadata but not found as folders: {len(cases_in_metadata_not_in_masks)}")
print(f"Case IDs as folders but not in metadata: {len(cases_in_masks_not_in_metadata)}")

print("\nExample metadata-positive/mask-negative cases:", list(metadata_says_positive_mask_says_not)[:10])
print("Example mask-positive/metadata-negative cases:", list(mask_says_positive_metadata_says_not)[:10])
print("Example metadata-only case IDs:", list(cases_in_metadata_not_in_masks)[:10])
print("Example mask-only case IDs:", list(cases_in_masks_not_in_metadata)[:10])


--- Discrepancy breakdown ---
Metadata says positive, mask says negative/missing: 44
Mask says positive, metadata says negative: 0
Case IDs in metadata but not found as folders: 0
Case IDs as folders but not in metadata: 1

Example metadata-positive/mask-negative cases: ['PanTS_00003142', 'PanTS_00006622', 'PanTS_00000676', 'PanTS_00005948', 'PanTS_00007898', 'PanTS_00005667', 'PanTS_00005836', 'PanTS_00000291', 'PanTS_00003770', 'PanTS_00006383']
Example mask-positive/metadata-negative cases: []
Example metadata-only case IDs: []
Example mask-only case IDs: ['.DS_Store']


# NOTES:
- Metadata indicates 1,077 tumour-positive cases; 
- However, 44 of these lack a corresponding non-empty lesion segmentation mask.
- They will be excluded from geometric label derivation since it requires a segmented lesion boundary.
- The effective tumour-positive cohort for this study is **1033 cases**.